# Predict Wine Quality with Regularization – Practice Skeleton

**Short name (GitHub):** `WineReg`  
**Lab source:** Codecademy *Predict Wine Quality with Regularization* (UCI red-wine table).  
**Data:** `data/wine_quality.csv` (1,599 bottles × 11 chemistry tests + binary `quality`).  
**Companion files:** `WineReg_Solution.ipynb`, `WineReg_Reusable_Template.ipynb`, `WineReg.py`, `WineReg_Cheatsheet.docx`, `WineReg_Project_Memo.docx`, `WineReg_Strategy_Guide.docx`, `WineReg_1Page_Summary_Report.docx`, `wine_reg_flowchart.png`.

Work top to bottom. Cells marked `# YOUR CODE HERE` are for you. Peek at the solution notebook only after you have an answer.

**Target.** Original ratings were 1–10. Here `quality = 1` means a *good* bottle (rating > 5) and `0` means *bad* (≤ 5). You will:

1. Fit an unregularized logistic classifier and read the coefficient bar.
2. Confirm that default L2 (`C=1`) barely shrinks anything.
3. Tune ridge `C` with a coarse grid, then `GridSearchCV`.
4. Use L1 (`LogisticRegressionCV`) as a feature-selection method — `density` should go to zero.


## Inline cheat-sheet (keep this cell visible)

See also **`WineReg_Cheatsheet.docx`**.

| Item | Code / rule |
|------|-------------|
| Scale | `StandardScaler().fit(features)` then `.transform(features)` |
| Split | `train_test_split(X, y, test_size=0.2, random_state=99)` |
| No penalty | `LogisticRegression(penalty=None, max_iter=2000)` (lesson: `'none'`; future: `C=np.inf`) |
| Default ridge | `LogisticRegression()` historically L2 with `C=1` |
| F1 | `f1_score(y_true, y_pred)` — harmonic mean of precision & recall |
| Coarse C | `[0.0001, 0.001, 0.01, 0.1, 1]` — smaller C = stronger L2 |
| Fine C | `np.logspace(-4, -2, 100)` |
| Grid search | `GridSearchCV(clf, {'C': C_array}, scoring='f1', cv=5)` |
| L1 CV | `LogisticRegressionCV(Cs=..., penalty='l1', solver='liblinear', scoring='f1')` |
| Coefs | `pd.Series(clf.coef_.ravel(), predictors).sort_values()` |
| C meaning | `C = 1 / λ`. Tiny C → shrink toward 0. Huge C → unregularized. |

**sklearn ≥1.8 note.** `penalty` is deprecated. Lesson code with `penalty='none'` still runs; prefer `penalty=None` or `C=np.inf` for no regularization, and set `penalty='l2'` / `'l1'` explicitly in this notebook so the intent stays readable.


## Flowchart of the desired outcome

![WineReg flow](wine_reg_flowchart.png)

Scale first (alcohol lives on a different scale than density). Split once and freeze `random_state=99`. Use F1, not raw accuracy — a 53.5% “good” prior makes accuracy a weak headline. Validate the “best” ridge model on the original test fold, then switch to L1 for sparsity.


## 0. Packages


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, classification_report


## 1. Load the table and separate X / y

UCI *Wine Quality* (red). Columns:

`fixed acidity`, `volatile acidity`, `citric acid`, `residual sugar`, `chlorides`,
`free sulfur dioxide`, `total sulfur dioxide`, `density`, `pH`, `sulphates`, `alcohol`, `quality`.

Print the columns, the class mix, and a short describe. Then set

* `y = df['quality']`
* `features = df.drop(columns=['quality'])`


In [ ]:
# YOUR CODE HERE
df = pd.read_csv("data/wine_quality.csv")
print(df.columns)
# inspect shape, quality value_counts, mean
y = None
features = None


## 2. Task 1 — scale with `StandardScaler`

Physicochemical tests do not share a unit. Alcohol (~8–15) would otherwise dominate density (~0.99). Fit the scaler on `features` and transform to `X`.


In [ ]:
# YOUR CODE HERE
standard_scaler_fit = None
X = None
print("X shape", getattr(X, "shape", None), "mean≈0?", None)


## 3. Task 2 — 80/20 split

`test_size=0.2`, `random_state=99` (lesson seed). Print train/test sizes and the good-wine rate in each fold — the seed happens to put a slightly different mix in the test fold.


In [ ]:
# YOUR CODE HERE
X_train, X_test, y_train, y_test = None, None, None, None


## 4. Task 3 — logistic regression *without* regularization

Define `clf_no_reg` with `penalty=None` (lesson text said `'none'`) and `max_iter=2000`. Fit on the training fold.


In [ ]:
# YOUR CODE HERE
clf_no_reg = None


## 5. Task 4 — coefficient bar (unregularized)

Copy the lesson snippet (or the equivalent `barh` version). Which three tests pull a bottle toward “good”? Which two pull toward “bad”?


In [ ]:
# YOUR CODE HERE
predictors = features.columns
coefficients = None
coef = None
# coef.plot(...)


## 6. Task 5 — training and test F1

Classifiers need more than accuracy. F1 is the harmonic mean of precision and recall. Compute both folds.


In [ ]:
# YOUR CODE HERE
y_pred_test = None
y_pred_train = None
print("Training Score", None)
print("Testing Score", None)


## 7. Task 6 — default implementation (historically L2, C=1)

`LogisticRegression()` with no arguments used to be ridge-regularized at `C=1`. Fit `clf_default` the same way the lesson does.


In [ ]:
# YOUR CODE HERE
clf_default = None


## 8. Task 7 — ridge F1 scores

Same metric as Task 5. Do the scores move?


In [ ]:
# YOUR CODE HERE
print("Ridge-regularized Training Score", None)
print("Ridge-regularized Testing Score", None)


## 9. Task 8 — coarse-grained C search

`C` is the *inverse* of regularization strength. To turn the ridge *up*, go below 1.

```
C_array = [0.0001, 0.001, 0.01, 0.1, 1]
```

Store train and test F1 in two lists.


In [ ]:
# YOUR CODE HERE
training_array = []
test_array = []
C_array = [0.0001, 0.001, 0.01, 0.1, 1]
for x in C_array:
    pass


## 10. Task 9 — plot F1 against C (log x)

```python
plt.plot(C_array, training_array, label="Training Score")
plt.plot(C_array, test_array, label="Test Score")
plt.xscale("log")
plt.xlabel("C")
plt.legend()
```

The interesting window is roughly `1e-4` to `1e-2`.


In [ ]:
# YOUR CODE HERE


## 11. Task 10 — parameter grid for `GridSearchCV`

`np.logspace(-4, -2, 100)` and wrap it as `tuning_C = {'C': C_array}`.


In [ ]:
# YOUR CODE HERE
C_array = None
tuning_C = None


## 12. Task 11 — `GridSearchCV` with L2 and F1

5-fold CV on the *training* fold only. Scoring = `'f1'`.


In [ ]:
# YOUR CODE HERE
clf_gs = None
gs = None


## 13. Task 12 — best C and its CV score

`gs.best_params_` and `gs.best_score_`.


In [ ]:
# YOUR CODE HERE


## 14. Task 13 — validate the “best classifier” on the original test fold

The CV score is an *in-train* estimate. Refit `clf_best` at that C and score F1 on `X_test`.


In [ ]:
# YOUR CODE HERE
clf_best = None
print(None)


## 15. Task 14 — L1 hyperparameter search with `LogisticRegressionCV`

Different API from `GridSearchCV`:

* `Cs` — array of C values, here `np.logspace(-2, 2, 100)`
* `cv=5`
* `penalty='l1'`
* `solver='liblinear'` (required for L1 in this solver family)
* `scoring='f1'`

Fit on **all** `(X, y)` as the lesson does (no second hold-out).


In [ ]:
# YOUR CODE HERE
clf_l1 = None


## 16. Task 15 — optimal L1 C and coefficients

`clf_l1.C_` and `clf_l1.coef_`.


In [ ]:
# YOUR CODE HERE
print("Best C value", None)
print("Best fit coefficients", None)


## 17. Task 16 — L1 coefficient bar

```python
coefficients = clf_l1.coef_.ravel()
coef = pd.Series(coefficients, predictors).sort_values()
coef.plot(kind="bar", title="Coefficients for tuned L1")
```


In [ ]:
# YOUR CODE HERE


## 18. Task 17 — what did L1 drop?

One coefficient should be (numerically) zero. That is Lasso doing feature selection.

Which test was eliminated? Why is that chemically plausible on this card?


In [ ]:
# YOUR CODE HERE
# print the name(s) whose |coef| is ~ 0


## 19. Alternate code (same scientific result)

Worked alternatives you can swap into the lesson cells.


In [ ]:
# YOUR CODE HERE — try at least one alternate
# A. LogisticRegression(C=np.inf) as the unregularized model
# B. Pipeline([StandardScaler(), LogisticRegression(...)]) on raw features
# C. LogisticRegressionCV for the L2 path instead of GridSearchCV
# D. A side-by-side coefficient table (unreg / ridge / l1)


## 20. More practice

### 20.1 Threshold as a cellar rule

`predict` uses 0.5. Sweep `t` in `{0.3, 0.4, 0.5, 0.6, 0.7}` on `clf_best` probabilities. Report F1, false-premium (FP: bad bottled as good) and missed-good (FN).


In [ ]:
# YOUR CODE HERE
proba = None


### 20.2 Three-test subset vs full card

Refit L2 (`C=0.002`) on only `alcohol`, `volatile acidity`, `total sulfur dioxide`. Compare hold-out F1 to the 11-test model.


In [ ]:
# YOUR CODE HERE


### 20.3 Class-weight balanced

The split is nearly even, but try `class_weight='balanced'` at the tuned C. Does recall of class 0 move?


In [ ]:
# YOUR CODE HERE


### 20.4 Elastic-net sketch (optional)

If your sklearn build accepts `penalty='elasticnet'` + `l1_ratio` + `solver='saga'`, try `l1_ratio=0.5` on a small C grid. Otherwise skip and note the deprecation path (`l1_ratio` on the new default estimator).


In [ ]:
# YOUR CODE HERE


## 21. Simulation — edit the knobs

Each replication: optional subsample, optional label flips, optional Gaussian junk columns, then L2 at `C_RIDGE` and L1 at `C_LASSO`. Watch how hold-out F1 and the number of exact-zero L1 weights move.


In [ ]:
# --- knobs ---
C_RIDGE = 0.002
C_LASSO = 0.26
N_SUB = 1279          # ≤ 1599; 1279 matches the lesson train size
FLIP_P = 0.00         # label-noise rate
N_JUNK = 0            # extra N(0,1) columns
N_REPS = 10
SEED = 7
# -------------

rng = np.random.default_rng(SEED)
rows = []
for r in range(N_REPS):
    n = min(N_SUB, len(X))
    idx = rng.choice(len(X), size=n, replace=False)
    Xb = X[idx].copy()
    yb = y.values[idx].copy()
    if FLIP_P > 0:
        flip = rng.random(n) < FLIP_P
        yb[flip] = 1 - yb[flip]
    if N_JUNK > 0:
        Xb = np.hstack([Xb, rng.normal(size=(n, N_JUNK))])
    Xt, Xv, yt, yv = train_test_split(Xb, yb, test_size=0.2, random_state=r)
    ridge = LogisticRegression(C=C_RIDGE, penalty="l2", max_iter=2000)
    ridge.fit(Xt, yt)
    lasso = LogisticRegression(C=C_LASSO, penalty="l1", solver="liblinear", max_iter=2000)
    lasso.fit(Xt, yt)
    rows.append({
        "rep": r,
        "ridge_f1": f1_score(yv, ridge.predict(Xv)),
        "lasso_f1": f1_score(yv, lasso.predict(Xv)),
        "lasso_zeros": int((np.abs(lasso.coef_.ravel()) < 1e-10).sum()),
    })
sim = pd.DataFrame(rows)
print(sim.round(3))
print(sim[["ridge_f1", "lasso_f1", "lasso_zeros"]].agg(["mean", "std"]).round(3))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].boxplot([sim["ridge_f1"], sim["lasso_f1"]], labels=["ridge", "lasso"])
ax[0].set_ylabel("hold-out F1")
ax[0].set_title(f"C_ridge={C_RIDGE}, C_lasso={C_LASSO}, flip={FLIP_P}, junk={N_JUNK}")
ax[1].hist(sim["lasso_zeros"], bins=range(0, 12), color="#1A5276", edgecolor="white")
ax[1].set_xlabel("# exact-zero L1 weights")
ax[1].set_title("sparsity across reps")
plt.tight_layout()
plt.show()


## 22. Audience rewrite (write your own, then compare to the solution)

Four cuts of the same result, using the attached audience checklists (data literacy, subject knowledge, time span, expert / technician / executive / nonspecialist).

**Expert (oenologist / statistician).**  
n = 1,599, binary cut at rating 5, prior π̂ = 0.535. Features standardized. Unpenalized MLE and L2 at C = 1 give the same hold-out F1 (0.727) — the L2 ball still contains the MLE. A 5-fold F1 grid on C ∈ [10⁻⁴, 10⁻²] selects C ≈ 0.002 (CV F1 ≈ 0.773); hold-out F1 = 0.735, AUC ≈ 0.81. L1-CV at C ≈ 0.26 zeros **density**, consistent with its linear dependence on alcohol / extract. Coefficients are log-odds per 1 SD. Do not treat the 5-cut as a sensory gold standard.

**Technician (lab / QC).**  
Scale the 11 tests. Fit logistic regression. If you leave C at the software default you have *not* regularized in any useful way. Set C near 0.002 for ridge, or use the L1 path and drop density from the daily card. Alcohol, sulphates, volatile acidity and total SO₂ do the work. F1 on a 320-bottle hold-out is about 0.73 — useful as a screen, not as a release stamp.

**Executive (winery GM / brand).**  
A chemistry-only model flags “good / not-good” at roughly three-in-four F1. Stronger shrinkage (smaller C) beats the default software settings. Density can leave the panel; alcohol and volatile acidity cannot. Use the probability threshold as a cellar rule: lower it to catch more good lots (more false premiums), raise it to protect the reserve label.

**Nonspecialist (curious drinker).**  
We asked 11 lab measurements to guess whether a red wine would have been scored above 5. Alcohol and a clean (not vinegary) profile help. Making the model “shy” about huge coefficients — regularization — stopped it from clinging to a redundant measurement (density) and slightly improved the guess on bottles it had not seen. It is a sorting aid, not a sommelier.


## 23. Takeaways

Write five bullets after you finish Tasks 1–17 and the simulation.